# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path(r"d:\Microsoft VS Code\date\E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()

读取文件：d:\Microsoft VS Code\date\E Commerce Dataset.xlsx
数据形状：5630 行，20 列


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [ ]:
df.info()

# 答案：
# 1.一行数据代表一位电商平台用户的全量行为、属性与消费相关的单条观测记录，包含该用户的注册时长、设备信息、消费情况、售后反馈等全部采集信息
# 2.该数据集里CustomerID（用户ID）字段是用户唯一标识
# 3Churn（用户流失)字段可作为用户流失分析的目标变量，该字段一般为0/1二值：1代表用户已流失、0代表用户仍留存

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAddress              

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [5]:
# TODO：创建 missing_report
# 提示：df.isna().sum() 统计缺失数量；df.isna().mean() 计算缺失比例。
missing_count = df.isnull().sum()
missing_ratio = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失比例(%)": missing_ratio
})
print("========== 缺失值统计报告 ==========")
print(missing_report)



# display(missing_report)

========== 缺失值统计报告 ==========
                             缺失数量  缺失比例(%)
CustomerID                      0     0.00
Churn                           0     0.00
Tenure                        264     4.69
PreferredLoginDevice            0     0.00
CityTier                        0     0.00
WarehouseToHome               251     4.46
PreferredPaymentMode            0     0.00
Gender                          0     0.00
HourSpendOnApp                255     4.53
NumberOfDeviceRegistered        0     0.00
PreferedOrderCat                0     0.00
SatisfactionScore               0     0.00
MaritalStatus                   0     0.00
NumberOfAddress                 0     0.00
Complain                        0     0.00
OrderAmountHikeFromlastYear   265     4.71
CouponUsed                    256     4.55
OrderCount                    258     4.58
DaySinceLastOrder             307     5.45
CashbackAmount                  0     0.00


### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [ ]:
# TODO：完成两项重复值统计
# 1. 统计完全重复的行数量
duplicate_rows = df.duplicated().sum()
# 2. 统计CustomerID出现重复的记录行数（同一个ID出现≥2次的全部条目）
duplicate_customer_ids = df[df["CustomerID"].duplicated(keep=False)].shape[0]

print("完全重复行数：", duplicate_rows)
print("CustomerID 重复数量：", duplicate_customer_ids)

# 思考：用户 ID 重复时，不能直接删除，因为重复CustomerID存在两种业务场景：
# 1. 同一用户有多条不同时间/行为的观测记录（多行为样本），直接删除会丢失用户行为数据，破坏样本完整性；
# 2. 只有当整行所有字段完全一致（属于脏数据重复录入），才可删除；仅ID重复但其他字段不同时，每条记录都具备独立业务含义，不可随意丢弃；
# 3. 需先区分重复类型：完全重复行可去重；ID重复但内容不同需业务确认是多行为样本还是采集错误，再做聚合/过滤处理，不能删除。


完全重复行数： 0
CustomerID 重复数量： 0


---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [7]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

...
for col in numeric_missing_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

# TODO：输出上述字段剩余的缺失值数量
# print(df[numeric_missing_cols].isna().sum())

C:\Users\王佳鑫\AppData\Local\Temp\ipykernel_36960\122367302.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(median_val, inplace=True)
C:\Users\王佳鑫\AppData\Local\Temp\ipykernel_36960\122367302.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an in

### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [ ]:
# 场景：数值字段存在明确业务含义的缺失，缺失本身带有信息
# 电商数据集 CashbackAmount（返现金额）字段
# 处理思路：① 衍生二元标记字段：HasCashback，缺失值填 0（无返现）、有数值填 1（有返现），保留缺失自带的业务信息；
#           ② 原CashbackAmount缺失值直接填充0

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [8]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [9]:
# TODO：完成类别标准化
if 'PreferredLoginDevice' in df.columns:
    # 统一登录设备：Phone、Mobile、Mobile Phone 全部标准化为 Mobile Phone
    df['PreferredLoginDevice'] = df['PreferredLoginDevice'].replace({
        "Phone": "Mobile Phone",
        "Mobile": "Mobile Phone"
    })
    print("\n===== 登录设备字段统一完成 =====")
    print(f"统一后设备类型取值: {df['PreferredLoginDevice'].unique()}")

if 'PreferredPaymentMode' in df.columns:
    # 统一支付方式：COD→Cash on Delivery；CC→Credit Card
    df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    })
    print("\n===== 支付方式字段统一完成 =====")
    print(f"统一后支付方式取值: {df['PreferredPaymentMode'].unique()}")


# TODO：重新检查类别频数
# for col in category_cols:
#     print(f"\n{col}")
#     print(df[col].value_counts())
category_cols = ["PreferredLoginDevice", "PreferredPaymentMode"]
for col in category_cols:
    print(f"\n【字段 {col} 分类频数统计】")
    print(df[col].value_counts())


===== 登录设备字段统一完成 =====
统一后设备类型取值: <StringArray>
['Mobile Phone', 'Computer']
Length: 2, dtype: str

===== 支付方式字段统一完成 =====
统一后支付方式取值: <StringArray>
['Debit Card', 'UPI', 'Credit Card', 'Cash on Delivery', 'E wallet']
Length: 5, dtype: str

【字段 PreferredLoginDevice 分类频数统计】
PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64

【字段 PreferredPaymentMode 分类频数统计】
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [11]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [ ]:
# TODO：调用函数检查三个字段

display(iqr_outlier_summary(df["WarehouseToHome"]))
display(iqr_outlier_summary(df["OrderCount"]))
display(iqr_outlier_summary(df["CashbackAmount"]))
# 结论：候选异常值不能直接删除，因为……
# 1. IQR仅从统计分布判定异常，不代表业务上是错误数据：
#    WarehouseToHome（仓库到家距离）偏大值可能是偏远地区的正常配送场景；
#    OrderCount（下单数）极高值可能是大客户/批发用户的真实消费行为；
#     CashbackAmount（返现金额）极值可能是平台大额活动的合规高额返现；
# 2. 直接删除会丢失高价值细分用户群体数据，造成样本偏差、后续流失分析/建模结果失真；
# 3. 正确做法：先结合业务核验异常成因，是采集错误才剔除修正，是真实业务数据则保留，可采用分箱、缩尾/截尾、单独建模等方式处理。

Q1         9.00
Q3        20.00
下限        -7.50
上限        36.50
候选异常值数量    2.00
dtype: float64

Q1          1.00
Q3          3.00
下限         -2.00
上限          6.00
候选异常值数量   703.00
dtype: float64

Q1        145.77
Q3        196.39
下限         69.84
上限        272.33
候选异常值数量   438.00
dtype: float64

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [ ]:

# TODO：完成业务规则检查
rules = {
    "使用时长小于 0": (df["HourSpendOnApp"] < 0).sum(),
    "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
    "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
}

rule_check_result = pd.Series(rules, name="违规记录条数")
print("===== 业务规则违规统计 =====")
print(rule_check_result)

===== 业务规则违规统计 =====
使用时长小于 0      0
仓库距离小于 0      0
订单数小于或等于 0    0
返现金额小于 0      0
Name: 违规记录条数, dtype: int64


---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [ ]:
# TODO：完成验收

numeric_missing_cols = [
    'Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress',
    'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed',
    'OrderCount', 'DaySinceLastOrder', 'CashbackAmount'
]

# print("数据清洗验收通过。")
numeric_missing_cols = [
    'Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress',
    'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed',
    'OrderCount', 'DaySinceLastOrder', 'CashbackAmount'
]

# 重新执行中位数填充
for col in numeric_missing_cols:
    # 转为数值，防止文本干扰
    df[col] = pd.to_numeric(df[col], errors="coerce")
    med_val = df[col].median()
    df[col].fillna(med_val, inplace=True)

# 二次校验缺失值
print("填充后总缺失量：", df[numeric_missing_cols].isna().sum().sum())

# 2. 校验输出目录是否存在，不存在则自动创建
output_dir = Path("./output")
if not output_dir.exists():
    output_dir.mkdir(parents=True)
    print(f"已创建输出目录：{output_dir}")




填充后总缺失量： 1856
已创建输出目录：output


C:\Users\王佳鑫\AppData\Local\Temp\ipykernel_36960\134561472.py:26: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(med_val, inplace=True)
C:\Users\王佳鑫\AppData\Local\Temp\ipykernel_36960\134561472.py:26: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inpla

### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [24]:
OUTPUT_PATH = Path("../output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO：导出为 UTF-8-SIG 编码的 CSV 文件
# df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")


print(f"已导出：{OUTPUT_PATH.resolve()}")
# print(f"已导出：{OUTPUT_PATH.resolve()}")

已导出：D:\Microsoft VS Code\output\ecommerce_customer_cleaned.csv


---
## 7. 提交前自查

- [ ] 已完成缺失值报告；
- [ ] 已完成重复记录检查；
- [ ] 已填补指定数值字段的缺失值；
- [ ] 已统一登录设备、支付方式和订单品类；
- [ ] 已完成候选异常值与业务规则检查；
- [ ] 已导出 ecommerce_customer_cleaned.csv；
- [ ] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？